# JUNE 25th-anniversary analysis — Step 1: build the article inventory

**What this does:** visits every issue page in the funjournal.org archive
(2002 through Spring 2025), adds the newer Scholastica issues (Fall 2025 on)
from CrossRef, and writes one row per published item to
`data/june_inventory.csv`.

**What you need to do:** run each cell from top to bottom
(in Google Colab: Runtime → Run all). The whole thing takes ~5 minutes
the first time. Web pages are saved in a `cache/` folder, so re-running
is fast and doesn't re-download anything.

**What comes out** (in the `data/` folder):
- `june_inventory.csv` — the master table (open it in Excel / Google Sheets)
- `june_inventory_flags.csv` — only the rows a human should double-check
- `june_issue_summary.csv` — number of items per issue and section, to spot gaps

**Next step:** `02_match_pubmed.ipynb` adds PubMed IDs, page numbers, and abstracts.

## Settings — the only cell you should normally need to edit

In [1]:
ARCHIVE_HOME = "https://www.funjournal.org/"

# Issue pages to add even if they're missing from the site's Archives menu.
EXTRA_ISSUE_URLS = []

# Issue pages to skip (e.g., empty placeholder pages).
SKIP_ISSUE_URLS = [
    "https://www.funjournal.org/2025-volume-25-issue-1/",  # empty template page
]

# Newer issues (~2020s) link to an article page rather than straight to the PDF.
# Those pages carry the DOI and abstract, so we visit them too.
FETCH_ARTICLE_PAGES = True

# FUN workshop proceedings issues, as (volume, issue). Every item in these
# issues starts out tagged as a workshop item; regular articles that happen to
# share the issue are corrected by hand later (see OPEN_QUESTIONS.md).
WORKSHOP_ISSUES = {
    (8, 1): "FUN 2008 Macalester workshop",
    (11, 1): "FUN 2011 Pomona workshop",
    (13, 3): "FUN 2014 Ithaca workshop",
    (16, 3): "FUN 2017 workshop",
    (20, 2): "FUN 2020 Summer Virtual Meeting",
    (22, 2): "FUN 2023 Western Washington workshop",
}

# From Fall 2025 (volume 24) on, JUNE is published on Scholastica. Those items
# come from CrossRef, where JUNE's publisher (FUN) is member 38466.
ADD_SCHOLASTICA_ITEMS = True
FIRST_SCHOLASTICA_VOLUME = 24
CROSSREF_MEMBER_ID = "38466"
CONTACT_EMAIL = ""   # optional: CrossRef asks for an email so they can reach you

SECONDS_BETWEEN_REQUESTS = 1.0   # be polite to the server
TRIES_PER_PAGE = 3               # the site sometimes drops connections; try again
CACHE_DIR = "cache"
DATA_DIR = "data"
OUTPUT_CSV = "data/june_inventory.csv"
FLAGS_CSV = "data/june_inventory_flags.csv"
SUMMARY_CSV = "data/june_issue_summary.csv"

## Setup (no need to edit below this line)

In [2]:
import hashlib
import os
import re
import time
from urllib.parse import urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup, NavigableString, Tag

HEADERS = {"User-Agent": "JUNE-25th-anniversary-inventory (academic, non-commercial)"}


def fetch_html(url):
    """Download a page, or read it from the cache if we've seen it before."""
    os.makedirs(CACHE_DIR, exist_ok=True)
    key = hashlib.md5(url.encode()).hexdigest()
    path = os.path.join(CACHE_DIR, key + ".html")
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return f.read()
    for attempt in range(1, TRIES_PER_PAGE + 1):
        time.sleep(SECONDS_BETWEEN_REQUESTS * attempt)   # wait a bit longer each try
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            resp.raise_for_status()
            break
        except requests.RequestException:
            if attempt == TRIES_PER_PAGE:
                raise      # give up; the caller prints the error
            print(f"  … retrying {url}")
    with open(path, "w", encoding="utf-8") as f:
        f.write(resp.text)
    return resp.text

## Turning a web page into simple lines of text

The issue pages were made by different people over 20+ years, so their HTML
varies a lot. Instead of relying on the HTML structure, we flatten each page
into lines that look like this:

    ## Articles
    [Title of the article](https://link-to-pdf)
    By Author A & Author B

and then read those lines in order, like a person would.

In [3]:
BLOCK_TAGS = {"p", "div", "li", "ul", "ol", "section", "article", "table",
              "tr", "td", "header", "footer", "nav", "blockquote"}
HEADING_TAGS = {"h1": "# ", "h2": "## ", "h3": "## ", "h4": "## "}


def html_to_lines(html):
    soup = BeautifulSoup(html, "lxml")
    for bad in soup(["script", "style", "noscript"]):
        bad.decompose()
    out = []

    def walk(node):
        if isinstance(node, NavigableString):
            out.append(str(node))
            return
        if not isinstance(node, Tag):
            return
        name = node.name
        if name == "br":
            out.append("\n")
            return
        if name in HEADING_TAGS:
            out.append("\n" + HEADING_TAGS[name] + node.get_text(" ", strip=True) + "\n")
            return
        if name == "a" and node.get("href"):
            text = node.get_text(" ", strip=True)
            out.append(f"[{text}]({node['href']})" if text else "")
            return
        if name in BLOCK_TAGS:
            out.append("\n")
        for child in node.children:
            walk(child)
        if name in BLOCK_TAGS:
            out.append("\n")

    walk(soup.body or soup)
    lines = [re.sub(r"\s+", " ", ln).strip() for ln in "".join(out).split("\n")]
    return [ln for ln in lines if ln]

## Step A: find every issue page from the site's Archives menu

In [4]:
ISSUE_LINK_RE = re.compile(
    r"^\[(?P<year>(19|20)\d\d),?\s+Volume\s+(?P<vol>\d+)\s*:?\s*Issue\s+(?P<iss>\d+)\]\((?P<url>[^)]+)\)$",
    re.I,
)


def find_issue_pages(home_lines):
    issues, seen = [], set()
    for ln in home_lines:
        m = ISSUE_LINK_RE.match(ln.lstrip("*- ").strip())
        if not m:
            continue
        url = m["url"]
        if not url.endswith("/"):
            url += "/"
        if url in seen or url in SKIP_ISSUE_URLS:
            continue
        seen.add(url)
        issues.append({"year_label": int(m["year"]), "volume": int(m["vol"]),
                       "issue": int(m["iss"]), "issue_url": url})
    return issues

## Step B: read one issue page into rows

Rules the parser follows:
- `## Something` starts a new section (Editorials, Articles, Media Reviews, …)
- a line with a link to a PDF or an article page starts a new item
- a line starting with "by" / "-by" / "By" right after it gives the authors
- in Media Reviews, "reviewed by" = the JUNE author, plain "by" = author
  of the book/media being reviewed
- links or lines mentioning "Supplementary" or "Movie" are counted as
  supplements, not separate items

In [5]:
LINK_RE = re.compile(r"\[([^\]]*)\]\(([^)]+)\)")
SUPPLEMENT_WORDS = re.compile(r"supplement|movie|video|appendix|download pdf", re.I)
PAGE_PREFIX_NAMES = {"A": "article", "E": "editorial", "R": "review",
                     "C": "case study", "T": "technical"}


def is_item_link(text, href):
    """True if this link points to a published item (not nav, not a supplement)."""
    # Check the link text AND the file name, e.g. "Instructions (PDF Document)"
    # pointing to Ramos_Supplemental_Material.pdf
    file_name = urlparse(href).path.split("/")[-1]
    if SUPPLEMENT_WORDS.search(text) or SUPPLEMENT_WORDS.search(file_name) \
            or re.search(r"-s\d{2,3}\.pdf$", href, re.I):
        return False
    path = urlparse(href).path.lower()
    if path.endswith(".pdf"):
        return True
    if "funjournal.org" not in href:
        return False
    # Newer issues link to an article page such as /volume-23-issue-2-.../name-june-232-a35-a43/
    if re.search(r"june-\d+[^/]*\d+/?$", path):
        return True
    # Some issues (e.g., 20(3)) link by WordPress ID instead: /?page_id=3641&preview=true
    # These redirect to the real article page, which we read later for page numbers.
    return re.search(r"[?&](page_id|p)=\d+", href) is not None


def canonical_url(html):
    """The page's official address (after any redirect), from <link rel="canonical">."""
    try:
        tag = BeautifulSoup(html, "lxml").find("link", rel="canonical")
    except Exception:
        return ""
    return tag["href"] if tag and tag.get("href") else ""


def subtype_from_title(title):
    """'AMAZING PAPERS IN NEUROSCIENCE: Teaching…' -> 'AMAZING PAPERS IN NEUROSCIENCE'."""
    m = re.match(r"^([A-Z][A-Z ’'&/-]{3,}[A-Z])\s*:", title)
    return m[1].strip() if m and m[1].strip() != "JUNE" else ""


def pages_from_url(href):
    """Pull the first (and, when available, last) page out of a PDF name or article URL."""
    file_name = urlparse(href).path.rstrip("/").split("/")[-1]
    last = file_name.lower()
    # Newer article pages: ...-june-232-a35-a43  or  ...-june-232e22-e25  or  a50-55
    m = re.search(r"june-\d+-?([a-z]?)(\d+)-([a-z]?)(\d+)$", last)
    if m:
        prefix = (m[1] or m[3] or "A").upper()
        return prefix, int(m[2]), int(m[4])
    # PDFs: june-12-e1.pdf, june-18-15.pdf, june-23-a35.pdf
    # (WordPress sometimes adds "-1" to repeated names: june-17-c1-1.pdf)
    m = re.search(r"june-\d+-([a-z]?)(\d+)(?:-\d)?\.pdf$", last)
    if m:
        return (m[1] or "A").upper(), int(m[2]), None
    # 2010-2012 PDFs name both pages: santer_9_1_a1_a4.pdf, berni_91_a5_a14proof.pdf
    m = re.search(r"[_-]([aecrt])(\d+)[_-]([aecrt])(\d+)", last)
    if m:
        return m[1].upper(), int(m[2]), int(m[4])
    # Early PDFs: LomE1.pdf, GittisA1.pdf (capital letter + page number).
    # Careful: birkett72.pdf or muir81.pdf end in volume+issue, not a page.
    # Many other early PDFs have no page in the name (KerchnerBlueprints.pdf).
    # That's OK: step 2 fills in pages from PubMed.
    m = re.search(r"([AECRT])(\d+)\.pdf$", file_name)
    if m:
        return m[1].upper(), int(m[2]), None
    return None, None, None


def clean_authors(text):
    text = re.sub(r"^[-–—\s]*(reviewed\s+)?by\s+", "", text, flags=re.I)
    return text.strip(" ,;")


def first_author_surname(authors):
    """Best guess at the first author's surname.
    Handles 'AG Gittis', 'CA Paul, EM Goergen', 'Stavnezer AJ', 'Ogilvie, JM', 'Flint, Jr. RW'."""
    if not authors:
        return ""
    first = re.split(r"\s*(?:&|\band\b|\bet al\.?)", authors)[0]
    first = first.split(",")[0].strip()          # first comma-separated piece
    words = [w for w in first.split() if w not in {"Jr.", "Jr", "Sr.", "III", "II"}]
    if not words:
        return ""
    is_initials = lambda w: re.fullmatch(r"[A-Z]{1,3}\.?", w) is not None
    if is_initials(words[0]) and len(words) > 1:   # "AG Gittis"
        return words[-1]
    return words[0]                                # "Stavnezer AJ" or "Ogilvie"


def surname_from_early_pdf(href):
    """Early PDFs are named after the first author, e.g. LomE1.pdf -> 'Lom'."""
    m = re.search(r"/([A-Za-z\-]+?)[AECRT]\d+\.pdf$", href or "")
    return m[1] if m and not m[1].lower().startswith("june") else ""


def parse_issue_lines(lines, issue_meta):
    # Keep only the issue's own content: from its title (# line) to the footer.
    start = next((i for i, ln in enumerate(lines) if ln.startswith("# ")), None)
    if start is None:
        start = next((i for i, ln in enumerate(lines) if ln.startswith("## ")), 0)
    end = next((i for i, ln in enumerate(lines) if i > start and ln.startswith("©")), len(lines))
    lines = lines[start:end]

    rows, section, pending_subtype, current = [], "", "", None
    for ln in lines:
        if ln.startswith("# "):
            continue
        if ln.startswith("## "):
            section, pending_subtype, current = ln[3:].strip(), "", None
            continue

        links = LINK_RE.findall(ln)
        item_links = [(t, h) for t, h in links if is_item_link(t, h)]

        if item_links:
            for title, href in item_links:
                # A long title is sometimes split into two links to the same
                # file, e.g. [Lights! Camera! Action Projects!](june-14-138.pdf)
                # [Engaging Psychopharmacology Students…](june-14-138.pdf).
                # That's one item, so join the titles instead of adding a row.
                if current is not None and not current["authors"] \
                        and href in (current["pdf_url"], current["article_page_url"]):
                    current["title"] += " " + title.strip()
                    continue
                before = ln.split("[" + title + "]")[0]
                prefix, p_start, p_end = pages_from_url(href)
                current = {
                    **issue_meta,
                    "section": section,
                    "subtype": pending_subtype or subtype_from_title(title.strip()),
                    "title": title.strip(),
                    "authors": "",
                    "reviewed_work_author": "",
                    "page_prefix": prefix,
                    "page_start": p_start,
                    "page_end": p_end,
                    "pdf_url": href if href.lower().endswith(".pdf") else "",
                    "article_page_url": "" if href.lower().endswith(".pdf") else href,
                    "n_supplements": 0,
                }
                # e.g. "Editorial by EP Wiertelak, Editor-in-Chief: [Title](...)"
                m = re.search(r"\bby\s+(.+?)(?:,\s*Editor.*)?:?\s*$", before, re.I)
                if m:
                    current["authors"] = m[1].strip(" :,")
                rows.append(current)
            pending_subtype = ""
            continue

        if current is None:
            # A short label line before an item, e.g. "Book Review"
            if not links and len(ln) < 40:
                pending_subtype = ln.strip("*_ ")
            continue

        # Supplements (linked or just italic text). The same line sometimes
        # names the authors too: "By S Kennedy [Supplementary Material](...)",
        # so remove the links and keep reading whatever text is left.
        if SUPPLEMENT_WORDS.search(ln):
            current["n_supplements"] += max(1, len(links))
            ln = LINK_RE.sub("", ln).strip()
            if not ln or SUPPLEMENT_WORDS.search(ln):
                continue
            links = []

        low = ln.lower().lstrip("-–— ")
        if low.startswith("reviewed by"):
            current["authors"] = clean_authors(ln)
        elif low.startswith("by "):
            if section.lower().startswith("media") and not current["reviewed_work_author"] \
                    and current["authors"] == "":
                # In media reviews, a plain "by" line names the book's author
                # — unless it's the only author line (fixed up below).
                current["reviewed_work_author"] = clean_authors(ln)
            elif not current["authors"]:
                current["authors"] = clean_authors(ln)
        elif not current["authors"] and not links and len(ln) < 300:
            # Newest pages sometimes list authors with no "By"
            current["authors"] = clean_authors(ln)
        elif not links and len(ln) < 40:
            pending_subtype = ln.strip("*_ ")

    for r in rows:
        # Media review with only a "by" line: that line is probably the reviewer.
        if r["section"].lower().startswith("media") and not r["authors"] and r["reviewed_work_author"]:
            r["authors"], r["reviewed_work_author"] = r["reviewed_work_author"], ""
        r["first_author"] = first_author_surname(r["authors"]) or surname_from_early_pdf(r["pdf_url"])
    return rows

## Step C (newer issues): read the article page for DOI, abstract, and PDF

In [6]:
DOI_RE = re.compile(r"10\.\d{4,9}/[^\s\"<>)*]+")


def parse_article_lines(lines):
    info = {"doi": "", "abstract": "", "pdf_url_from_page": ""}
    text = "\n".join(lines)
    m = DOI_RE.search(text)
    if m:
        info["doi"] = m[0].rstrip(".,;")
    abs_start = next((i for i, ln in enumerate(lines) if ln.strip().upper() == "ABSTRACT"), None)
    if abs_start is not None:
        body = []
        for ln in lines[abs_start + 1:]:
            if ln.startswith("[") or ln.startswith("©") or ln.upper().startswith("KEY WORDS"):
                break
            body.append(ln)
        info["abstract"] = " ".join(body).strip()
    for t, h in LINK_RE.findall(text):
        if h.lower().endswith(".pdf"):
            info["pdf_url_from_page"] = h
            break
    return info

## Step D: quality checks

Rather than silently trusting the parser, every row gets a `flags` column
listing anything a human should look at. Nothing is deleted.

In [7]:
def add_flags(df):
    flags = [[] for _ in range(len(df))]
    expected_prefix = {"editorial": "E", "article": "A", "media": "R", "case": "C", "technical": "T"}
    for i, r in enumerate(df.itertuples()):
        if not r.authors:
            flags[i].append("no authors found")
        if r.source != "funjournal.org":
            continue   # Scholastica items have no page numbers, PDF links, or sections here
        if r.page_start is None or pd.isna(r.page_start):
            flags[i].append("no page number")
        sec = str(r.section).lower()
        for key, pre in expected_prefix.items():
            if sec.startswith(key) and r.page_prefix and r.page_prefix != pre:
                flags[i].append(f"section '{r.section}' but page prefix {r.page_prefix}")
        if not r.pdf_url:
            flags[i].append("no PDF link")
        if r.section == "":
            flags[i].append("no section heading")
    df["flags"] = ["; ".join(f) for f in flags]

    dup = df.duplicated(subset=["volume", "page_prefix", "page_start"], keep=False) & df["page_start"].notna()
    df.loc[dup, "flags"] = df.loc[dup, "flags"].apply(lambda s: (s + "; " if s else "") + "duplicate volume+page")
    return df


def scholastica_items_from_crossref():
    """Items from Fall 2025 on (Scholastica), one dict per item, from CrossRef.

    Not cached, so each run picks up newly published issues."""
    headers = dict(HEADERS)
    if CONTACT_EMAIL:
        headers["User-Agent"] += f" (mailto:{CONTACT_EMAIL})"
    works, offset = [], 0
    while True:
        resp = requests.get(
            "https://api.crossref.org/works",
            params={"filter": f"member:{CROSSREF_MEMBER_ID}", "rows": 100, "offset": offset},
            headers=headers, timeout=60)
        resp.raise_for_status()
        page = resp.json()["message"]["items"]
        works.extend(page)
        if len(page) < 100:
            break
        offset += 100
        time.sleep(SECONDS_BETWEEN_REQUESTS)

    rows = []
    for w in works:
        journal = " ".join(w.get("container-title", []))
        if "Neuroscience Education" not in journal:
            continue
        try:
            volume, issue = int(w.get("volume", 0)), int(w.get("issue", 0))
        except ValueError:
            continue
        if volume < FIRST_SCHOLASTICA_VOLUME:
            continue      # older items are already on funjournal.org
        authors = ", ".join(f"{a.get('given', '')} {a.get('family', '')}".strip()
                            for a in w.get("author", []))
        # CrossRef abstracts contain XML tags like <jats:p>; strip them.
        abstract = re.sub(r"<[^>]+>", " ", w.get("abstract", ""))
        abstract = re.sub(r"^\s*Abstract\s+", "", re.sub(r"\s+", " ", abstract)).strip()
        rows.append({
            "year_label": w.get("published", {}).get("date-parts", [[None]])[0][0],
            "volume": volume, "issue": issue,
            "section": "", "subtype": "",
            "title": " ".join(w.get("title", [])).strip(),
            "authors": authors,
            "first_author": (w.get("author") or [{}])[0].get("family", ""),
            "reviewed_work_author": "",
            "page_prefix": None, "page_start": None, "page_end": None,
            "doi": w.get("DOI", ""),
            "pdf_url": "",
            "article_page_url": w.get("resource", {}).get("primary", {}).get("URL", ""),
            "issue_url": "",
            "n_supplements": 0,
            "abstract": abstract,
            "source": "CrossRef (Scholastica)",
        })
    rows.sort(key=lambda r: (r["volume"], r["issue"], r["title"]))
    return rows


def issue_completeness(issues):
    """List any volume whose issue numbers aren't 1, 2, (3) with no gaps."""
    problems = []
    by_vol = {}
    for it in issues:
        by_vol.setdefault(it["volume"], set()).add(it["issue"])
    for v in range(1, max(by_vol) + 1):
        got = by_vol.get(v, set())
        if not got:
            problems.append(f"Volume {v}: no issues found")
        elif sorted(got) != list(range(1, max(got) + 1)):
            problems.append(f"Volume {v}: issues found = {sorted(got)}")
    return problems

## Run everything

In [8]:
def build_inventory():
    home_lines = html_to_lines(fetch_html(ARCHIVE_HOME))
    issues = find_issue_pages(home_lines)
    for url in EXTRA_ISSUE_URLS:
        issues.append({"year_label": None, "volume": None, "issue": None, "issue_url": url})
    print(f"Found {len(issues)} issue pages.")
    for p in issue_completeness([i for i in issues if i["volume"]]):
        print("  ⚠", p)

    rows = []
    for n, meta in enumerate(sorted(issues, key=lambda d: (d["volume"] or 0, d["issue"] or 0)), 1):
        try:
            lines = html_to_lines(fetch_html(meta["issue_url"]))
        except Exception as e:
            print(f"  ✗ could not read {meta['issue_url']}: {e}")
            continue
        items = parse_issue_lines(lines, meta)
        print(f"[{n}/{len(issues)}] Vol {meta['volume']}({meta['issue']}) {meta['year_label']}: {len(items)} items")
        rows.extend(items)

    df = pd.DataFrame(rows)
    df["doi"], df["abstract"] = "", ""
    df["source"] = "funjournal.org"

    if FETCH_ARTICLE_PAGES:
        todo = df.index[df["article_page_url"] != ""]
        print(f"Reading {len(todo)} article pages for DOIs/abstracts…")
        for idx in todo:
            try:
                html = fetch_html(df.at[idx, "article_page_url"])
                info = parse_article_lines(html_to_lines(html))
            except Exception as e:
                print(f"  ✗ {df.at[idx, 'article_page_url']}: {e}")
                continue
            real_url = canonical_url(html)
            if real_url and real_url != df.at[idx, "article_page_url"]:
                df.at[idx, "article_page_url"] = real_url   # replace ?page_id= links
            if pd.isna(df.at[idx, "page_start"]) or df.at[idx, "page_start"] is None:
                prefix, p_start, p_end = pages_from_url(real_url)
                if p_start is None:
                    prefix, p_start, p_end = pages_from_url(info["pdf_url_from_page"])
                df.at[idx, "page_prefix"], df.at[idx, "page_start"], df.at[idx, "page_end"] = prefix, p_start, p_end
            df.at[idx, "doi"] = info["doi"]
            df.at[idx, "abstract"] = info["abstract"]
            if not df.at[idx, "pdf_url"]:
                df.at[idx, "pdf_url"] = info["pdf_url_from_page"]

    if ADD_SCHOLASTICA_ITEMS:
        try:
            new = scholastica_items_from_crossref()
            print(f"Added {len(new)} Scholastica items (volume {FIRST_SCHOLASTICA_VOLUME}+) from CrossRef.")
            df = pd.DataFrame(df.to_dict("records") + new)
        except Exception as e:
            print(f"  ✗ could not read CrossRef: {e}")

    df["page_start"] = df["page_start"].astype("Int64")
    df["page_end"] = df["page_end"].astype("Int64")
    df["item_type_from_page"] = df["page_prefix"].map(PAGE_PREFIX_NAMES)

    # A unique ID for every row: volume, issue, and position in the issue,
    # e.g. JUNE-04-2-07 = volume 4, issue 2, 7th item listed.
    # (Page numbers can't be used: many early items don't have one here.)
    position = df.groupby(["volume", "issue"]).cumcount() + 1
    df["item_id"] = [f"JUNE-{v:02d}-{i}-{p:02d}" for v, i, p in zip(df["volume"], df["issue"], position)]

    # FUN workshop proceedings (see WORKSHOP_ISSUES at the top)
    workshop = [WORKSHOP_ISSUES.get((v, i), "") for v, i in zip(df["volume"], df["issue"])]
    df["workshop_name"] = workshop
    df["is_fun_workshop"] = [w != "" for w in workshop]

    df = add_flags(df)

    cols = ["item_id", "year_label", "volume", "issue", "section", "subtype",
            "item_type_from_page", "is_fun_workshop", "workshop_name",
            "title", "authors", "first_author",
            "reviewed_work_author", "page_prefix", "page_start", "page_end",
            "doi", "pdf_url", "article_page_url", "issue_url",
            "n_supplements", "abstract", "source", "flags"]
    df = df[cols]
    os.makedirs(DATA_DIR, exist_ok=True)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")  # utf-8-sig opens cleanly in Excel
    df[df["flags"] != ""].to_csv(FLAGS_CSV, index=False, encoding="utf-8-sig")
    (df.replace({"section": {"": "(not listed)"}})
       .groupby(["year_label", "volume", "issue", "section"]).size()
       .unstack(fill_value=0).to_csv(SUMMARY_CSV, encoding="utf-8-sig"))

    print(f"\nDone: {len(df)} items → {OUTPUT_CSV}")
    print(f"{(df['flags'] != '').sum()} rows flagged for a human check → {FLAGS_CSV}")
    return df


inventory = build_inventory()
inventory.head(20)

Found 50 issue pages.
[1/50] Vol 1(1) 2002: 9 items
[2/50] Vol 1(2) 2003: 9 items
[3/50] Vol 2(1) 2003: 7 items
[4/50] Vol 2(2) 2004: 9 items
[5/50] Vol 3(1) 2004: 7 items
[6/50] Vol 3(2) 2005: 9 items
[7/50] Vol 4(1) 2005: 10 items
[8/50] Vol 4(2) 2006: 9 items
[9/50] Vol 5(1) 2006: 10 items
[10/50] Vol 5(2) 2007: 13 items
[11/50] Vol 6(1) 2007: 9 items
[12/50] Vol 6(2) 2008: 9 items
[13/50] Vol 7(1) 2008: 7 items
[14/50] Vol 7(2) 2009: 10 items
[15/50] Vol 8(1) 2009: 19 items
[16/50] Vol 8(2) 2010: 9 items
[17/50] Vol 9(1) 2010: 11 items
[18/50] Vol 9(2) 2011: 9 items
[19/50] Vol 10(1) 2011: 13 items
[20/50] Vol 10(2) 2012: 9 items
[21/50] Vol 11(1) 2012: 28 items
[22/50] Vol 11(2) 2013: 6 items
[23/50] Vol 12(1) 2013: 17 items
[24/50] Vol 12(2) 2014: 13 items
[25/50] Vol 13(1) 2014: 14 items


[26/50] Vol 13(2) 2015: 12 items
[27/50] Vol 13(3) 2015: 18 items
[28/50] Vol 14(1) 2015: 19 items
[29/50] Vol 14(2) 2016: 20 items
[30/50] Vol 15(1) 2016: 29 items
[31/50] Vol 15(2) 2017: 18 items
[32/50] Vol 16(1) 2017: 23 items
[33/50] Vol 16(2) 2018: 23 items
[34/50] Vol 16(3) 2018: 21 items
[35/50] Vol 17(1) 2018: 19 items
[36/50] Vol 17(2) 2019: 11 items
[37/50] Vol 18(1) 2019: 10 items
[38/50] Vol 18(2) 2020: 10 items
[39/50] Vol 19(1) 2020: 21 items
[40/50] Vol 19(2) 2021: 14 items
[41/50] Vol 20(1) 2021: 15 items
[42/50] Vol 20(2) 2022: 26 items
[43/50] Vol 20(3) 2022: 14 items
[44/50] Vol 21(1) 2022: 11 items
[45/50] Vol 21(2) 2023: 14 items
[46/50] Vol 22(1) 2023: 13 items
[47/50] Vol 22(2) 2024: 16 items
[48/50] Vol 22(3) 2024: 16 items
[49/50] Vol 23(1) 2024: 8 items
[50/50] Vol 23(2) 2025: 11 items
Reading 129 article pages for DOIs/abstracts…


Added 21 Scholastica items (volume 24+) from CrossRef.

Done: 708 items → data/june_inventory.csv
129 rows flagged for a human check → data/june_inventory_flags.csv


,item_id,year_label,volume,issue,section,subtype,item_type_from_page,is_fun_workshop,workshop_name,title,...,page_start,page_end,doi,pdf_url,article_page_url,issue_url,n_supplements,abstract,source,flags
0,JUNE-01-1-01,2002,1,1,Editorials,,editorial,False,,Introducing The Journal of Undergraduate Neuro...,...,1,<NA>,,https://www.funjournal.org/wp-content/uploads/...,,https://www.funjournal.org/2002-volume-1-issue-1/,0,,funjournal.org,no authors found
1,JUNE-01-1-02,2002,1,1,Articles,,article,False,,The Emergence of Non-Match-to-Sample Behavior ...,...,1,<NA>,,https://www.funjournal.org/wp-content/uploads/...,,https://www.funjournal.org/2002-volume-1-issue-1/,0,,funjournal.org,
2,JUNE-01-1-03,2002,1,1,Articles,,article,False,,And The Winner Is: Inviting Hollywood into the...,...,4,<NA>,,https://www.funjournal.org/wp-content/uploads/...,,https://www.funjournal.org/2002-volume-1-issue-1/,0,,funjournal.org,
3,JUNE-01-1-04,2002,1,1,Articles,,article,False,,Exploring Neurogenesis in Crustaceans,...,18,<NA>,,https://www.funjournal.org/wp-content/uploads/...,,https://www.funjournal.org/2002-volume-1-issue-1/,2,,funjournal.org,
4,JUNE-01-1-05,2002,1,1,Articles,•,article,False,,Action Potentials in a Giant Algal Cell: A Com...,...,23,<NA>,,https://www.funjournal.org/wp-content/uploads/...,,https://www.funjournal.org/2002-volume-1-issue-1/,0,,funjournal.org,
5,JUNE-01-1-06,2002,1,1,Media Reviews,,review,False,,Book Review: Mood Genes,...,1,<NA>,,https://www.funjournal.org/wp-content/uploads/...,,https://www.funjournal.org/2002-volume-1-issue-1/,0,,funjournal.org,
6,JUNE-01-1-07,2002,1,1,Media Reviews,,review,False,,Textbook Review: Biological Psychology,...,3,<NA>,,https://www.funjournal.org/wp-content/uploads/...,,https://www.funjournal.org/2002-volume-1-issue-1/,0,,funjournal.org,
7,JUNE-01-1-08,2002,1,1,Media Reviews,,review,False,,Media Review: Crawdad: A CD-ROM Lab Manual for...,...,5,<NA>,,https://www.funjournal.org/wp-content/uploads/...,,https://www.funjournal.org/2002-volume-1-issue-1/,0,,funjournal.org,
8,JUNE-01-1-09,2002,1,1,Media Reviews,,review,False,,Media Review: The Secret Life of the Brain,...,7,<NA>,,https://www.funjournal.org/wp-content/uploads/...,,https://www.funjournal.org/2002-volume-1-issue-1/,0,,funjournal.org,
9,JUNE-01-2-01,2003,1,2,Editorials,,editorial,False,,Introductory Neuroscience – Courses in an Evol...,...,2,<NA>,,https://www.funjournal.org/wp-content/uploads/...,,https://www.funjournal.org/2003-volume-1-issue-2/,0,,funjournal.org,


## Check the results

These tables are for a human sanity check. Things to look for:
- a volume with far fewer (or more) items than its neighbors
- items in the wrong section (e.g., an editorial counted as an article)
- workshop items in an issue that isn't a workshop issue, or vice versa

Scholastica items (volume 24+) show "(not listed)" as their section,
because CrossRef doesn't record sections.

In [9]:
# Items per volume, by section. Workshop items are counted in their own column.
check = inventory.replace({"section": {"": "(not listed)"}})
per_volume = (check[~check["is_fun_workshop"]]
              .groupby(["volume", "section"]).size().unstack(fill_value=0))
per_volume["FUN workshop items"] = check[check["is_fun_workshop"]].groupby("volume").size()
per_volume = per_volume.fillna(0).astype(int)
per_volume["total"] = per_volume.sum(axis=1)
per_volume

section,(not listed),Articles,Case Studies,Editorials,Interview,Media Reviews,Media Reviews and Technical Reports,Technical Papers,FUN workshop items,total
volume,,,,,,,,,,
1,0,8,0,3,0,7,0,0,0,18
2,0,8,0,3,0,5,0,0,0,16
3,0,11,0,3,0,2,0,0,0,16
4,0,11,0,6,0,2,0,0,0,19
5,0,11,0,7,0,5,0,0,0,23
6,0,12,0,4,0,2,0,0,0,18
7,0,12,0,4,0,1,0,0,0,17
8,0,7,0,1,1,0,0,0,19,28
9,0,15,0,5,0,0,0,0,0,20


In [10]:
# FUN workshop issues: how many items in each
(inventory[inventory["is_fun_workshop"]]
 .groupby(["volume", "issue", "workshop_name"]).size().rename("items").reset_index())

,volume,issue,workshop_name,items
0,8,1,FUN 2008 Macalester workshop,19
1,11,1,FUN 2011 Pomona workshop,28
2,13,3,FUN 2014 Ithaca workshop,18
3,16,3,FUN 2017 workshop,21
4,20,2,FUN 2020 Summer Virtual Meeting,26
5,22,2,FUN 2023 Western Washington workshop,16


In [11]:
# What kinds of problems were flagged, and how often
inventory["flags"].str.split("; ").explode().loc[lambda s: s != ""].value_counts()

flags
no page number                          123
duplicate volume+page                     4
no authors found                          2
section 'Articles' but page prefix R      1
Name: count, dtype: int64